In [8]:
import numpy as np

# 1. The Embedding Matrix (E)
# Shape: (V=5, d=4)
# Rows represent: 0:"the", 1:"dog", 2:"sat", 3:"on", 4:"mat"
E = np.array([
    [ 0.1,  0.2, -0.1,  0.3],
    [ 0.4, -0.2,  0.5,  0.1],
    [-0.3,  0.1,  0.2, -0.4],
    [ 0.2,  0.5,  0.1, -0.2],
    [-0.1, -0.3,  0.4,  0.2]
])

EMBEDDING_VOCAB_SIZE = np.size(E, axis=0)
EMBEDDING_DIMENSION = np.size(E, axis=1)

# 2. The Input Tokens
# Our sentence: "the dog sat" -> indices [0, 1, 2]
tokens = np.array([0, 1, 2])

# 3. The Learned Weight Matrices (Initialized randomly)
# Shape: (d=4, d_k=4) for all three.
W_Q = np.array([
    [ 0.2, -0.1,  0.3,  0.0],
    [ 0.4,  0.1, -0.2,  0.3],
    [-0.1,  0.5,  0.2, -0.1],
    [ 0.3,  0.0,  0.1,  0.4]
])

W_K = np.array([
    [ 0.1,  0.3, -0.2,  0.1],
    [-0.2,  0.0,  0.4,  0.2],
    [ 0.3, -0.1,  0.1,  0.3],
    [ 0.0,  0.4, -0.3,  0.1]
])

W_V = np.array([
    [ 0.3,  0.1,  0.2, -0.1],
    [ 0.1, -0.2,  0.3,  0.4],
    [-0.2,  0.3,  0.0,  0.1],
    [ 0.4,  0.0, -0.1,  0.2]
])

In [11]:
# Constructing X (Ingestion Layer)

# Since we have three tokens, lets get the value of index = 0 to test
v = np.array([1,0,0,0,0])
emb_tok_1 = E.T @ v.T
print(emb_tok_1)

# Lets generate programmatically
print("\n Generate programmatically")
x_list = []

for i in tokens:
    v_i = np.zeros(EMBEDDING_VOCAB_SIZE)
    v_i[i] = 1

    x_i = E.T @ v_i.T ## Formula is E.T @ v
    
    # Note dimension of x_i = (d . V @ V . 1 = d . 1)
    # Transpose x_1 leads to dimension 1 .d
    # X is stack of these matrics
    x_list.append(x_i.T.tolist())

X = np.array(x_list)
print(X)

[ 0.1  0.2 -0.1  0.3]

 Generate programmatically
[[ 0.1  0.2 -0.1  0.3]
 [ 0.4 -0.2  0.5  0.1]
 [-0.3  0.1  0.2 -0.4]]


In [12]:
# Numpy trick 

X_NEW = E[tokens]
print(X_NEW)

[[ 0.1  0.2 -0.1  0.3]
 [ 0.4 -0.2  0.5  0.1]
 [-0.3  0.1  0.2 -0.4]]


In [13]:
Q = X @ W_Q
K = X @ W_K
V = X @ W_V

print(Q)
print(K)
print(V)

[[ 2.00000000e-01 -4.00000000e-02 -1.04083409e-17  1.90000000e-01]
 [-2.00000000e-02  1.90000000e-01  2.70000000e-01 -7.00000000e-02]
 [-1.60000000e-01  1.40000000e-01 -1.10000000e-01 -1.50000000e-01]]
[[-0.06  0.16 -0.04  0.05]
 [ 0.23  0.11 -0.14  0.16]
 [ 0.01 -0.27  0.24  0.01]]
[[ 0.19 -0.06  0.05  0.12]
 [ 0.04  0.23  0.01 -0.05]
 [-0.28  0.01  0.01  0.01]]


In [15]:
import math
# Calculate attention scores now

S = Q @ K.T
print(S)

# Scale S

S = S / (math.sqrt(EMBEDDING_DIMENSION))
print("After scaling")
print(S)

[[-0.0089  0.072   0.0147]
 [ 0.0173 -0.0327  0.0126]
 [ 0.0289 -0.03   -0.0673]]
After scaling
[[-0.00445  0.036    0.00735]
 [ 0.00865 -0.01635  0.0063 ]
 [ 0.01445 -0.015   -0.03365]]


In [26]:
## Now, let's apply Softmax

# Softmax is applied per row

S_NORMALISED_LIST = []

for row in S:
    row_e_power = [math.exp(x) for x in row]
    sum_e_power = sum(row_e_power)
    row_e_power_weighted = [x/sum_e_power for x in row_e_power]
    S_NORMALISED_LIST.append(row_e_power_weighted)

A = np.array(S_NORMALISED_LIST)
print(A)

[[0.32753068 0.34105089 0.33141843]
 [0.3363648  0.32805993 0.33557527]
 [0.34199517 0.33207027 0.32593456]]


In [27]:
## Now, above normalization but using Numpy tricks

S_max_subtracted = S - np.max(S, axis=1, keepdims=True)
S_max_subtracted_pow = np.exp(S_max_subtracted)

S_max_subtracted_pow_sum = np.sum(S_max_subtracted_pow, axis=1, keepdims=True)
A_NEW = S_max_subtracted_pow / S_max_subtracted_pow_sum

print(A_NEW)

[[0.32753068 0.34105089 0.33141843]
 [0.3363648  0.32805993 0.33557527]
 [0.34199517 0.33207027 0.32593456]]


In [32]:
# Final Attention output
ATTENTION_OUTPUT = A @ V
print(ATTENTION_OUTPUT)

[[-0.0169243   0.06210405  0.02310123  0.02556532]
 [-0.01692937  0.05862765  0.02345459  0.02731653]
 [-0.01299978  0.0591158   0.02367981  0.02769525]]
